KG Incidenti

In [1]:
import pandas as pd
from rdflib import Graph, Literal, Namespace, RDF, RDFS, URIRef
from rdflib.namespace import XSD

# Definizione dei Namespace
EX = Namespace("http://tesi.example.org/ontology/")
NIS2 = Namespace("http://tesi.example.org/nis2/")


def slugify(text: str) -> str:
  """Normalizza stringhe per creare URI RDF sicuri."""
  if not text or pd.isna(text):
    return "Unknown"
  return (
      "".join(c if c.isalnum() else "_" for c in str(text).strip())
      .strip("_")
      .title()
  )


def build_incidents_subgraph(
    df_incidenti: pd.DataFrame, base_graph: Graph = None
) -> Graph:
  g = base_graph if base_graph is not None else Graph()
  g.bind("ex", EX)
  g.bind("nis2", NIS2)
  g.bind("xsd", XSD)

  for _, row in df_incidenti.iterrows():
    inc_id = str(row["id"]).strip()
    company_name = str(row["denominazione"]).strip()
    dimensione = (
        str(row["dimensione"]).strip().upper()
        if pd.notna(row["dimensione"])
        else "N_D"
    )
    settore = str(row["settore"]).strip() if pd.notna(row["settore"]) else "N_D"
    data_incidente = str(row["data"]).strip()  # Formato atteso: YYYY-MM-DD
    minaccia = (
        str(row["minaccia"]).strip() if pd.notna(row["minaccia"]) else "Generica"
    )

    # Definizione URIs
    incident_uri = EX[f"Incident_{slugify(inc_id)}"]
    company_uri = EX[f"Company_{slugify(company_name)}"]
    sector_uri = EX[f"Sector_{slugify(settore)}"]
    threat_uri = EX[f"Threat_{slugify(minaccia)}"]

    # 1. Triplette Azienda
    g.add((company_uri, RDF.type, EX.Company))
    g.add(
        (
            company_uri,
            EX.denominazione,
            Literal(company_name, datatype=XSD.string),
        )
    )
    g.add((company_uri, EX.dimensioneAziendale, EX[f"Dimensione_{dimensione}"]))
    g.add((company_uri, EX.hasAssignedSector, sector_uri))
    g.add((company_uri, EX.hasIncident, incident_uri))

    # 2. Triplette Settore
    g.add((sector_uri, RDF.type, EX.Sector))
    g.add((sector_uri, RDFS.label, Literal(settore, datatype=XSD.string)))

    # 3. Triplette Minaccia
    g.add((threat_uri, RDF.type, EX.Threat))
    g.add((threat_uri, RDFS.label, Literal(minaccia, datatype=XSD.string)))

    # 4. Triplette Incidente
    g.add((incident_uri, RDF.type, EX.SecurityIncident))
    g.add((incident_uri, EX.incidentId, Literal(inc_id, datatype=XSD.string)))
    g.add((incident_uri, EX.targetsCompany, company_uri))
    g.add((incident_uri, EX.involvesSector, sector_uri))
    g.add((incident_uri, EX.hasThreat, threat_uri))

    # Data normalizzata xsd:date
    if data_incidente and data_incidente != "nan":
      g.add(
          (
              incident_uri,
              EX.incidentDate,
              Literal(data_incidente, datatype=XSD.date),
          )
      )

  return g

In [2]:
from pyvis.network import Network


def export_incident_network(graph: Graph, output_html: str = "incidents_kg.html"):
  net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white")
  net.barnes_hut()

  query = """
    PREFIX ex: <http://tesi.example.org/ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?companyName ?sectorName ?threatName ?incId
    WHERE {
      ?incident a ex:SecurityIncident ;
                ex:incidentId ?incId ;
                ex:targetsCompany ?company ;
                ex:involvesSector ?sector ;
                ex:hasThreat ?threat .

      ?company ex:denominazione ?companyName .
      ?sector rdfs:label ?sectorName .
      ?threat rdfs:label ?threatName .
    }
    """

  # Palette nodi
  # Aziende: Grigio/Azzurro | Settori: Verde/Blu | Minacce: Rosso | Incidente: Giallo
  for row in graph.query(query):
    comp = str(row.companyName)
    sec = str(row.sectorName)
    threat = str(row.threatName)
    inc = f"INC-{str(row.incId)}"

    net.add_node(comp, label=comp, color="#97C2FC", shape="box")
    net.add_node(sec, label=sec, color="#7BE141", shape="ellipse")
    net.add_node(threat, label=threat, color="#FB7E81", shape="diamond")
    net.add_node(inc, label=inc, color="#FFCC00", shape="dot", size=10)

    net.add_edge(comp, sec, title="belongsToSector")
    net.add_edge(inc, comp, title="targets")
    net.add_edge(inc, threat, title="threatType")

  net.save_graph(output_html)
  print(f"Grafo visualizzato salvato in {output_html}")

In [3]:
import pandas as pd
from rdflib import Graph

# Carica il dataset pulito
df_inc = pd.read_excel("data/dataset_incidenti_sicurezza_enisa_mitre.xlsx")

# Popola il grafo RDF
g = Graph()
build_incidents_subgraph(df_inc, base_graph=g)

# Serializza in formato Turtle per GraphDB o persistenza locale
g.serialize(destination="knowledge_graph_incidenti.ttl", format="turtle")
print(f"Grafo aggiornato con successo: {len(g)} triplette create.")

Grafo aggiornato con successo: 490 triplette create.


Correlazione dimensione minaccia

In [4]:
import pandas as pd
from pyvis.network import Network
from rdflib import Graph



# 2. Query aggregata per estrarre la matrice di esposizione
query_agg = """
PREFIX ex: <http://tesi.example.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?settore ?minaccia (COUNT(?incident) AS ?peso)
WHERE {
  ?incident a ex:SecurityIncident ;
            ex:involvesSector ?sec ;
            ex:hasThreat ?thr .
  ?sec rdfs:label ?settore .
  ?thr rdfs:label ?minaccia .
}
GROUP BY ?settore ?minaccia
"""

net = Network(
    height="750px", width="100%", bgcolor="#1a1a1a", font_color="#f0f0f0"
)
net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=150)

# 3. Costruzione della rete PyVis
settori_inseriti = set()
minacce_inserite = set()

for row in g.query(query_agg):
  sec = str(row.settore)
  thr = str(row.minaccia).split("(")[0].strip()  # Abbreviazione label pulita
  peso = int(row.peso)

  if sec not in settori_inseriti:
    net.add_node(
        sec,
        label=sec,
        color="#4CAF50",
        shape="dot",
        size=22,
        title=f"Settore: {sec}",
    )
    settori_inseriti.add(sec)

  if thr not in minacce_inserite:
    net.add_node(
        thr,
        label=thr,
        color="#F44336",
        shape="diamond",
        size=18,
        title=f"Minaccia: {str(row.minaccia)}",
    )
    minacce_inserite.add(thr)

  net.add_edge(
      sec,
      thr,
      value=peso,
      title=f"Incidenti rilevati: {peso}",
      color="#888888",
  )

net.save_graph("matrice_rischio_settori_minacce.html")
print("Visualizzazione generata: 'matrice_rischio_settori_minacce.html'")

Visualizzazione generata: 'matrice_rischio_settori_minacce.html'
